In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "fable_data"

def get_silver_transactions(catalog, schema):
    return spark.table(
        f"{catalog}.{schema}.silver_transactions"
    )

def get_valid_transactions(silver_df):
    return silver_df.filter(
    F.col("TRANSACTION_AMOUNT").isNotNull()
    & F.col("TRANSACTION_DATE").isNotNull()
    & F.col("CUSTOMER_KEY").isNotNull()
)


In [0]:
def standard_metrics():
    return [
        F.count("*").alias("TRANSACTION_COUNT"),
        F.round(
            F.sum("TRANSACTION_AMOUNT"),
            2
        ).alias("TOTAL_TRANSACTION_AMOUNT"),
        F.round(
            F.avg("TRANSACTION_AMOUNT"),
            2
        ).alias("AVERAGE_TRANSACTION_AMOUNT"),
        F.countDistinct(
            "CUSTOMER_KEY"
        ).alias("UNIQUE_CUSTOMERS"),
    ]

In [0]:
def get_daily_transaction_metrics(silver_df):
    return (
    silver_df
    .groupBy("TRANSACTION_DATE")
    .agg(*standard_metrics())
    .orderBy("TRANSACTION_DATE")
    )

def get_country_metrics(silver_df):
    return (
    silver_df
    .groupBy("TRANSACTION_DATE", "COUNTRY_CODE")
    .agg(*standard_metrics())
    .orderBy("COUNTRY_CODE")
    )

def get_age_band_metrics(silver_df):
    return (
    silver_df
    .groupBy("TRANSACTION_DATE", "AGE_BAND")
    .agg(*standard_metrics())
    .orderBy("AGE_BAND")
    )

def get_gender_metrics(silver_df):
    return (
    silver_df
    .groupBy("TRANSACTION_DATE", "GENDER_CODE")
    .agg(*standard_metrics())
    .orderBy("GENDER_CODE")
    )


In [0]:

def generate_gold_dataframes(silver_df):
    valid_trans_df = get_valid_transactions(silver_df)
    daily_metrics_df = get_daily_transaction_metrics(valid_trans_df)
    country_metrics_df = get_country_metrics(valid_trans_df)
    age_band_metrics_df = get_age_band_metrics(valid_trans_df)
    gender_metrics_df = get_gender_metrics(valid_trans_df)
    return daily_metrics_df, country_metrics_df, age_band_metrics_df, gender_metrics_df

def write_gold_dataframes(catalog, schema, daily_metrics_df, country_metrics_df, age_band_metrics_df, gender_metrics_df):
    daily_metrics_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.gold_daily_metrics")
    country_metrics_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.gold_country_metrics")
    age_band_metrics_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.gold_age_band_metrics")
    gender_metrics_df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.{schema}.gold_gender_metrics")


In [0]:


daily_metrics_df, country_metrics_df, age_band_metrics_df, gender_metrics_df = generate_gold_dataframes(get_silver_transactions(CATALOG, SCHEMA))
write_gold_dataframes(CATALOG, SCHEMA, daily_metrics_df, country_metrics_df, age_band_metrics_df, gender_metrics_df)